In [ ]:
import glob, re, os, zipfile, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
if os.path.basename(os.getcwd()) == 'for_exam': os.chdir('..')
if not os.path.isdir('data_exp8'): zipfile.ZipFile('data_exp8.zip').extractall()

files = glob.glob('data_exp8/Summaries/**/*.txt', recursive=True)
df = pd.DataFrame({'file': [os.path.basename(f) for f in files], 'text': [open(f, encoding='utf-8').read() for f in files]})
print('Loaded', len(df), 'articles')

In [ ]:
df['clean'] = df['text'].apply(lambda t: ' '.join(w for w in re.findall(r'[a-z]+', t.lower()) if w not in ENGLISH_STOP_WORDS and len(w) > 2))
vec = CountVectorizer(max_df=0.9, min_df=2)
X = vec.fit_transform(df['clean'])
terms = vec.get_feature_names_out()
print('Matrix shape:', X.shape)

In [ ]:
freq = np.asarray(X.sum(axis=0)).ravel()
top = freq.argsort()[-20:][::-1]
plt.figure(figsize=(12, 5)); plt.bar(terms[top], freq[top])
plt.xticks(rotation=60); plt.title('Top 20 Frequent Terms'); plt.tight_layout(); plt.show()

In [ ]:
lda = LatentDirichletAllocation(n_components=5, max_iter=20, random_state=42).fit(X)
for i, topic in enumerate(lda.components_):
    print(f'Topic {i+1}: ' + ', '.join(terms[topic.argsort()[-10:][::-1]]))

topics = pd.DataFrame(lda.transform(X), columns=[f'Topic_{i+1}' for i in range(5)])
topics.insert(0, 'file', df['file'])
topics.head(10).round(3)